# Phase 3 — SQL Analytics (Python + PostgreSQL)

Runs **12 analytics queries** against Phase 1 data in **`master_data`**, using view **`analytics.master_enriched`**.

**Prerequisite:** Phase 1 loaded `public.master_data`. The notebook creates the analytics view automatically (same DDL as `phase3_analytics_setup.sql`).

```bash
pip install pandas sqlalchemy psycopg2-binary
```

## Setup — Imports and helpers

In [23]:
from __future__ import annotations

import sys
from pathlib import Path
from typing import Dict

import pandas as pd
from sqlalchemy import create_engine, inspect, text
from sqlalchemy.engine import Engine
from sqlalchemy.exc import SQLAlchemyError

PROJECT_DIR = Path.cwd().resolve()
MASTER_TABLE = "master_data"
ANALYTICS_VIEW = "analytics.master_enriched"
print(f"Project directory: {PROJECT_DIR}")

Project directory: D:\NCPL\Bootcamp\Projects\Project 5\new data\data 2


In [24]:
def prompt_db_credentials() -> Dict[str, str]:
    return {
        "host": input("PostgreSQL host: ").strip(),
        "port": input("PostgreSQL port (5432): ").strip() or "5432",
        "database": input("Database name: ").strip(),
        "username": input("Username: ").strip(),
        "password": input("Password: ").strip(),
    }


def create_pg_engine(creds: Dict[str, str]) -> Engine:
    missing = [k for k in ("host", "database", "username", "password") if not creds.get(k)]
    if missing:
        raise ValueError(f"Missing connection fields: {missing}")
    url = (
        f"postgresql+psycopg2://{creds['username']}:{creds['password']}"
        f"@{creds['host']}:{creds['port']}/{creds['database']}"
    )
    return create_engine(url, pool_pre_ping=True)


def test_connection(engine: Engine) -> None:
    with engine.connect() as conn:
        print(conn.execute(text("SELECT version()")).scalar())
    print("Connection successful.")


def _analytics_view_exists(engine: Engine) -> bool:
    with engine.connect() as conn:
        return bool(
            conn.execute(
                text(
                    "SELECT EXISTS ("
                    "  SELECT 1 FROM information_schema.views"
                    "  WHERE table_schema = 'analytics' AND table_name = 'master_enriched'"
                    ")"
                )
            ).scalar()
        )


def run_phase3_analytics_setup(engine: Engine) -> None:
    """Create analytics schema + master_enriched view (from phase3_analytics_setup.sql)."""
    setup_path = PROJECT_DIR / "phase3_analytics_setup.sql"
    if not setup_path.is_file():
        raise FileNotFoundError(f"Missing setup file: {setup_path}")

    sql_text = setup_path.read_text(encoding="utf-8")
    marker = "CREATE OR REPLACE VIEW analytics.master_enriched"
    start = sql_text.find(marker)
    if start < 0:
        raise RuntimeError(
            f"No {marker} statement found in {setup_path.name}"
        )
    end = sql_text.find(";", start)
    if end < 0:
        raise RuntimeError(f"Malformed CREATE VIEW statement in {setup_path.name}")
    view_stmt = sql_text[start : end + 1]

    with engine.begin() as conn:
        conn.execute(text("CREATE SCHEMA IF NOT EXISTS analytics"))
        conn.execute(text(view_stmt))
    print(f"Created {ANALYTICS_VIEW} from {setup_path.name}.")


def verify_phase3_setup(engine: Engine) -> None:
    insp = inspect(engine)
    if not insp.has_table(MASTER_TABLE, schema="public"):
        raise RuntimeError(
            f"public.{MASTER_TABLE} not found. Run Phase 1 notebook first."
        )
    if not _analytics_view_exists(engine):
        run_phase3_analytics_setup(engine)
    with engine.connect() as conn:
        n = conn.execute(text(f"SELECT COUNT(*) FROM {ANALYTICS_VIEW}")).scalar()
    print(f"public.{MASTER_TABLE} and {ANALYTICS_VIEW} OK ({n:,} rows)")


def run_query(engine: Engine, sql: str, name: str) -> pd.DataFrame:
    try:
        df = pd.read_sql(text(sql), engine)
    except SQLAlchemyError as exc:
        print(f"[{name}] Error: {exc}", file=sys.stderr)
        raise
    if df.empty:
        print(f"[{name}] Warning: 0 rows returned.")
    else:
        print(f"[{name}] {len(df):,} rows × {len(df.columns)} cols")
    return df

---
## Step 1 — Connect to PostgreSQL

In [25]:
try:
    engine = create_pg_engine(prompt_db_credentials())
    test_connection(engine)
    verify_phase3_setup(engine)
except (SQLAlchemyError, ValueError, RuntimeError) as exc:
    print(f"Setup error: {exc}", file=sys.stderr)
    raise

PostgreSQL 18.3 on x86_64-windows, compiled by msvc-19.44.35225, 64-bit
Connection successful.
public.master_data and analytics.master_enriched OK (1,581,466 rows)


---
## Step 2 — Load master data

Preview **`master_data`** (Phase 1) and the enriched analytics view.

In [26]:
try:
    df_master = pd.read_sql(text(f"SELECT * FROM public.{MASTER_TABLE} LIMIT 5"), engine)
    df_enriched = pd.read_sql(text(f"SELECT * FROM {ANALYTICS_VIEW} LIMIT 5"), engine)
except SQLAlchemyError as exc:
    print(f"Load error: {exc}", file=sys.stderr)
    raise

with engine.connect() as conn:
    total = conn.execute(text(f"SELECT COUNT(*) FROM public.{MASTER_TABLE}")).scalar()

print(f"master_data rows: {total:,}")
print(f"master_data columns: {list(df_master.columns)}")
print(f"enriched columns:    {list(df_enriched.columns)}")
display(df_master.head())
display(df_enriched[["row_id", "product_name", "sentiment", "rating", "region", "review_text"]].head())

master_data rows: 1,581,466
master_data columns: ['row_id', 'target', 'ids', 'date', 'flag', 'user', 'text']
enriched columns:    ['row_id', 'user', 'target', 'ids', 'date', 'flag', 'text', 'review_text', 'product_name', 'category', 'brand', 'region', 'sentiment', 'rating', 'created_at']


,row_id,target,ids,date,flag,user,text
0,1,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,2,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,3,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,4,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,5,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


,row_id,product_name,sentiment,rating,region,review_text
0,1,Sentiment 0,negative,0,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,2,Sentiment 0,negative,0,scotthamilton,is upset that he can't update his Facebook by ...
2,3,Sentiment 0,negative,0,mattycus,@Kenichan I dived many times for the ball. Man...
3,4,Sentiment 0,negative,0,ElleCTF,my whole body feels itchy and like its on fire
4,5,Sentiment 0,negative,0,Karoli,"@nationwideclass no, it's not behaving at all...."


---
## Query 1 — Total reviews by product

Sentiment class (`target`) treated as product category.

In [27]:
SQL_Q1 = """
SELECT sentiment, category, COUNT(*) AS total_reviews
FROM analytics.master_enriched
GROUP BY sentiment, category
ORDER BY total_reviews DESC, sentiment
"""
q1 = run_query(engine, SQL_Q1, "Query 1")
display(q1)

[Query 1] 2 rows × 3 cols


,sentiment,category,total_reviews
0,positive,4,791281
1,negative,0,790185


## Query 2 — Sentiment count by product

In [28]:
SQL_Q2 = """
SELECT category,
    COUNT(*) FILTER (WHERE sentiment = 'positive') AS positive_count,
    COUNT(*) FILTER (WHERE sentiment = 'neutral')  AS neutral_count,
    COUNT(*) FILTER (WHERE sentiment = 'negative') AS negative_count,
    COUNT(*) AS total_reviews
FROM analytics.master_enriched
GROUP BY product_name, category
ORDER BY total_reviews DESC
"""
q2 = run_query(engine, SQL_Q2, "Query 2")
display(q2)

[Query 2] 2 rows × 5 cols


,category,positive_count,neutral_count,negative_count,total_reviews
0,4,791281,0,0,791281
1,0,0,0,790185,790185


## Query 3 — Sentiment trend by week and month

In [29]:
SQL_Q3 = """
WITH weekly AS (
    SELECT DATE_TRUNC('week', created_at)::DATE AS period_start, 'week' AS period_grain,
           sentiment, COUNT(*) AS review_count
    FROM analytics.master_enriched WHERE created_at IS NOT NULL
    GROUP BY 1, sentiment
),
monthly AS (
    SELECT DATE_TRUNC('month', created_at)::DATE AS period_start, 'month' AS period_grain,
           sentiment, COUNT(*) AS review_count
    FROM analytics.master_enriched WHERE created_at IS NOT NULL
    GROUP BY 1, sentiment
)
SELECT * FROM weekly UNION ALL SELECT * FROM monthly
ORDER BY period_grain, period_start, sentiment
"""
q3 = run_query(engine, SQL_Q3, "Query 3")
display(q3.head(15))

[Query 3] 29 rows × 4 cols


,period_start,period_grain,sentiment,review_count
0,2009-04-01,month,negative,41324
1,2009-04-01,month,positive,57944
2,2009-05-01,month,negative,204294
3,2009-05-01,month,positive,322463
4,2009-06-01,month,negative,544567
5,2009-06-01,month,positive,410874
6,2009-04-06,week,negative,8557
7,2009-04-06,week,positive,12021
8,2009-04-13,week,negative,19537
9,2009-04-13,week,positive,27640


## Query 4 — Negative reviews by category

Count of negative tweets grouped by sentiment class (`target`).


In [30]:
SQL_Q4 = """
SELECT category, COUNT(*) AS negative_review_count
FROM analytics.master_enriched
WHERE sentiment = 'negative'
GROUP BY category
ORDER BY negative_review_count DESC, category
"""
q4 = run_query(engine, SQL_Q4, "Query 4")
display(q4)


[Query 4] 1 rows × 2 cols


,category,negative_review_count
0,0,790185


## Query 5 — Top complaint topics

Keyword search on `review_text` (`text` column from master_data).

In [31]:
SQL_Q5 = """
WITH complaint_keywords AS (
    SELECT unnest(ARRAY['bad', 'poor', 'broken', 'refund', 'delay']) AS keyword
),
matched AS (
    SELECT ck.keyword, m.row_id
    FROM analytics.master_enriched m
    CROSS JOIN complaint_keywords ck
    WHERE m.review_text IS NOT NULL
      AND LOWER(m.review_text) LIKE '%' || ck.keyword || '%'
)
SELECT keyword, COUNT(DISTINCT row_id) AS review_mentions, COUNT(*) AS keyword_occurrences
FROM matched GROUP BY keyword ORDER BY review_mentions DESC
"""
q5 = run_query(engine, SQL_Q5, "Query 5")
display(q4)

[Query 5] 5 rows × 3 cols


,category,negative_review_count
0,0,790185


## Query 6 — Sentiment by user (`user` column)

Top 50 users by tweet volume with sentiment breakdown.


In [32]:
SQL_Q6 = """
SELECT region,
    COUNT(*) FILTER (WHERE sentiment = 'positive') AS positive_count,
    COUNT(*) FILTER (WHERE sentiment = 'neutral')  AS neutral_count,
    COUNT(*) FILTER (WHERE sentiment = 'negative') AS negative_count,
    COUNT(*) AS total_reviews
FROM analytics.master_enriched
GROUP BY region
ORDER BY total_reviews DESC, region
LIMIT 50
"""
q6 = run_query(engine, SQL_Q6, "Query 6")
display(q6.head(15))


[Query 6] 50 rows × 5 cols


,region,positive_count,neutral_count,negative_count,total_reviews
0,lost_dog,0,0,549,549
1,webwoke,81,0,260,341
2,SallytheShizzle,98,0,183,281
3,VioletsCRUK,218,0,61,279
4,mcraddictal,65,0,209,274
5,tsarnick,212,0,36,248
6,what_bugs_u,246,0,0,246
7,Karen230683,118,0,119,237
8,DarkPiano,229,0,5,234
9,SongoftheOss,128,0,99,227


## Query 7 — Product rating vs sentiment

Average rating and sentiment mix per sentiment class.


In [33]:
SQL_Q7 = """
WITH stats AS (
    SELECT product_name, category, AVG(rating) AS avg_rating,
        COUNT(*) FILTER (WHERE sentiment = 'positive') AS positive_count,
        COUNT(*) FILTER (WHERE sentiment = 'neutral')  AS neutral_count,
        COUNT(*) FILTER (WHERE sentiment = 'negative') AS negative_count,
        COUNT(*) AS total_reviews
    FROM analytics.master_enriched
    GROUP BY product_name, category
)
SELECT product_name, category,
    ROUND(avg_rating::NUMERIC, 2) AS avg_rating,
    positive_count, neutral_count, negative_count, total_reviews,
    ROUND(100.0 * positive_count / NULLIF(total_reviews, 0), 2) AS positive_pct,
    ROUND(100.0 * neutral_count  / NULLIF(total_reviews, 0), 2) AS neutral_pct,
    ROUND(100.0 * negative_count / NULLIF(total_reviews, 0), 2) AS negative_pct
FROM stats
ORDER BY avg_rating DESC NULLS LAST, total_reviews DESC
"""
q7 = run_query(engine, SQL_Q7, "Query 7")
display(q7)


[Query 7] 2 rows × 10 cols


,product_name,category,avg_rating,positive_count,neutral_count,negative_count,total_reviews,positive_pct,neutral_pct,negative_pct
0,Sentiment 4,4,4.0,791281,0,0,791281,100.0,0.0,0.0
1,Sentiment 0,0,0.0,0,0,790185,790185,0.0,0.0,100.0


## Query 8 — Most reviewed products (top 10)

Sentiment classes with the highest tweet volume.


In [34]:
SQL_Q8 = """
SELECT product_name, category, brand, COUNT(*) AS review_count
FROM analytics.master_enriched
GROUP BY product_name, category, brand
ORDER BY review_count DESC, product_name
LIMIT 10
"""
q8 = run_query(engine, SQL_Q8, "Query 8")
display(q8)


[Query 8] 2 rows × 4 cols


,product_name,category,brand,review_count
0,Sentiment 4,4,NO_QUERY,791281
1,Sentiment 0,0,NO_QUERY,790185


## Query 9 — Latest negative reviews

In [35]:
SQL_Q9 = """
SELECT row_id, product_name, review_text, created_at, "user"
FROM analytics.master_enriched
WHERE sentiment = 'negative'
ORDER BY created_at DESC NULLS LAST, row_id DESC
LIMIT 20
"""
q9 = run_query(engine, SQL_Q9, "Query 9")
display(q5)

[Query 9] 20 rows × 5 cols


,keyword,review_mentions,keyword_occurrences
0,bad,30119,30119
1,poor,8612,8612
2,broken,3617,3617
3,delay,1152,1152
4,refund,127,127


## Query 10 — Brand-wise sentiment (`flag` column)

In [36]:
SQL_Q10 = """
WITH brand_stats AS (
    SELECT brand, COUNT(*) AS total_reviews,
        COUNT(*) FILTER (WHERE sentiment = 'positive') AS positive_count,
        COUNT(*) FILTER (WHERE sentiment = 'neutral')  AS neutral_count,
        COUNT(*) FILTER (WHERE sentiment = 'negative') AS negative_count
    FROM analytics.master_enriched GROUP BY brand
)
SELECT brand, total_reviews,
    ROUND(100.0 * positive_count / NULLIF(total_reviews, 0), 2) AS positive_pct,
    ROUND(100.0 * negative_count / NULLIF(total_reviews, 0), 2) AS negative_pct,
    ROUND(100.0 * neutral_count  / NULLIF(total_reviews, 0), 2) AS neutral_pct
FROM brand_stats ORDER BY total_reviews DESC
"""
q10 = run_query(engine, SQL_Q10, "Query 10")
display(q6)

[Query 10] 1 rows × 5 cols


,region,positive_count,neutral_count,negative_count,total_reviews
0,lost_dog,0,0,549,549
1,webwoke,81,0,260,341
2,SallytheShizzle,98,0,183,281
3,VioletsCRUK,218,0,61,279
4,mcraddictal,65,0,209,274
5,tsarnick,212,0,36,248
6,what_bugs_u,246,0,0,246
7,Karen230683,118,0,119,237
8,DarkPiano,229,0,5,234
9,SongoftheOss,128,0,99,227


## Query 11 — Average rating by category

In [37]:
SQL_Q11 = """
SELECT category, ROUND(AVG(rating)::NUMERIC, 2) AS avg_rating, COUNT(*) AS review_count
FROM analytics.master_enriched
WHERE rating IS NOT NULL
GROUP BY category ORDER BY avg_rating DESC NULLS LAST
"""
q11 = run_query(engine, SQL_Q11, "Query 11")
display(q7)

[Query 11] 2 rows × 3 cols


,product_name,category,avg_rating,positive_count,neutral_count,negative_count,total_reviews,positive_pct,neutral_pct,negative_pct
0,Sentiment 4,4,4.0,791281,0,0,791281,100.0,0.0,0.0
1,Sentiment 0,0,0.0,0,0,790185,790185,0.0,0.0,100.0


## Query 12 — Daily review volume (last 90 days)

In [38]:
SQL_Q12 = """
WITH bounds AS (
    SELECT MAX(created_at) AS max_created_at
    FROM analytics.master_enriched WHERE created_at IS NOT NULL
),
daily AS (
    SELECT DATE_TRUNC('day', m.created_at)::DATE AS review_day, COUNT(*) AS review_count
    FROM analytics.master_enriched m
    CROSS JOIN bounds b
    WHERE m.created_at IS NOT NULL
      AND m.created_at >= b.max_created_at - INTERVAL '90 days'
    GROUP BY 1
)
SELECT review_day, review_count FROM daily ORDER BY review_day
"""
q12 = run_query(engine, SQL_Q12, "Query 12")
display(q12.tail(15))

[Query 12] 40 rows × 2 cols


,review_day,review_count
25,2009-06-05,15209
26,2009-06-06,97498
27,2009-06-07,110894
28,2009-06-08,44466
29,2009-06-15,60637
30,2009-06-16,100171
31,2009-06-17,43754
32,2009-06-18,40042
33,2009-06-19,43461
34,2009-06-20,43497


---
## Summary

In [39]:
summary = pd.DataFrame({
    "query": [f"Q{i}" for i in range(1, 13)],
    "description": [
        "Reviews by product/sentiment class",
        "Sentiment counts by class",
        "Weekly/monthly trend",
        "Negative by category",
        "Complaint keywords",
        "Sentiment by user",
        "Rating vs sentiment",
        "Top 10 classes",
        "Latest negative reviews",
        "Sentiment by flag/brand",
        "Avg rating by category",
        "Daily volume (90d)",
    ],
    "rows": [len(q1), len(q2), len(q3), len(q4), len(q5), len(q6),
             len(q7), len(q8), len(q9), len(q10), len(q11), len(q12)],
})
display(summary)
print("Phase 3 complete.")

,query,description,rows
0,Q1,Reviews by product/sentiment class,2
1,Q2,Sentiment counts by class,2
2,Q3,Weekly/monthly trend,29
3,Q4,Negative by category,1
4,Q5,Complaint keywords,5
5,Q6,Sentiment by user,50
6,Q7,Rating vs sentiment,2
7,Q8,Top 10 classes,2
8,Q9,Latest negative reviews,20
9,Q10,Sentiment by flag/brand,1


Phase 3 complete.
